# 06: Anchor-residue ablation experiment

## The biological question

An MHC class I peptide has two functional roles, encoded in different residues:

1. **MHC binding.** Anchor positions (P2 and PΩ) sit inside MHC pockets B and F. The amino acid identity at these positions is the dominant determinant of whether the peptide is presented at all. These residues are buried; the T-cell receptor never sees them directly.
2. **TCR engagement.** Solvent-exposed positions (roughly P4 through P7 in a 9-mer) face outward from the MHC groove and form the actual contact surface for the TCR. The amino acid identity here determines whether a presented peptide is *recognised* and triggers a T-cell response.

Our dataset labels peptides as immunogenic (1) or not (0) based on whether they elicited a T-cell response in an assay. A positive label therefore requires *both* (a) the peptide bound MHC well enough to be presented, and (b) the TCR engaged it. A negative label can come from failure of either.

Notebook 05 turned up a suggestive finding: the strongest *negative* LR coefficients at C-terminal positions are exactly the canonical MHC-I anchor residues (L, I, M, Y, K). One reading is that the negatives in IEDB are enriched for peptides that bind MHC well but failed downstream, meaning what we're calling "immunogenicity prediction" may be partly capturing the inverse of MHC binding strength.

## The experiment

Train the same three models on **two views of the same data**:

- **Full sequence**: the existing one-hot encoding (notebook 02 baseline).
- **TCR-contact only**: the same encoding with anchor positions zeroed out.

Compare AUC on the held-out test set. Three possible outcomes:

| Outcome                                | Biological reading                                                               |
|----------------------------------------|----------------------------------------------------------------------------------|
| Masked AUC >> full AUC                 | Anchor positions were adding noise; TCR-contact residues carry the real signal.  |
| Masked AUC ≈ full AUC                  | The signal is robust, distributed across positions, or both encodings work.     |
| Masked AUC << full AUC                 | Models depended heavily on anchors. "Immunogenicity" prediction is largely MHC binding prediction in this dataset. |

This is a small but real piece of *intentional science*. The same pipeline runs twice; the difference is what we choose to encode.

## What is filled in for you, and what you do

**Filled in:** environment setup, the masking helper, references back to the saved feature matrix, and the comparison-table scaffolding. The retraining of LR and RF takes about 30 seconds; the CNN takes a few minutes.

**You complete:** two `TODO` cells. The first builds the masked feature matrix. The second runs the three trainings on the masked data and fills out the comparison table. Both are marked clearly. The point of doing it yourself is that *the masking step is the experiment*, and writing it teaches you how to manipulate one-hot encodings.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

MAX_LEN, N_AA = 11, 21
RANDOM_STATE  = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = Path('..') / 'data' / 'processed'

## Load the saved features

We start from the same `features.npz` produced in notebook 02. The features are flat (231-dim) but laid out as 11 positions × 21 amino acids; we'll reshape, mask, then flatten.

In [2]:
data = np.load(DATA_DIR / 'features.npz')
X_train_full, X_test_full = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

X_train_3d = X_train_full.reshape(-1, MAX_LEN, N_AA)
X_test_3d  = X_test_full.reshape(-1, MAX_LEN, N_AA)
print(f'Full input: train {X_train_3d.shape}, test {X_test_3d.shape}')

Full input: train (38081, 11, 21), test (9521, 11, 21)


## Defining the anchor positions to mask

There is a **subtlety**. The encoding in notebook 02 right-pads to length 11. So:

- For a **9-mer**, the anchors are at sequence positions P2 and P9. After right-padding into the 11-slot grid, those land at column index 1 and column index 8.
- For an **8-mer**, PΩ is the 8th residue, landing at column index 7.
- For a **10-mer**, PΩ lands at column index 9.
- For an **11-mer**, PΩ lands at column index 10.

Right-padding scatters the C-terminal anchor across columns 7-10 depending on length. The simplest way to define the experiment cleanly is to **restrict the comparison to 9-mers only**, where anchor positions are unambiguous. That is what we do below.

Alternative (more work, more rigorous): re-encode with center-padding before doing the ablation. We flag this as a stretch goal at the end.

In [3]:
# Identify 9-mers in the existing encoding by counting non-padding positions.
# In the (n, 11, 21) array, column 20 is the padding token. A 9-mer has the
# padding column lit at the last 2 positions (positions 10 and 11, 0-indexed 9 and 10).

def is_nmer(X_3d, n):
    """Return a boolean mask of rows in X_3d that have exactly n non-padding positions."""
    non_pad_count = (X_3d[:, :, 20] == 0).sum(axis=1)
    return non_pad_count == n

train_9mer = is_nmer(X_train_3d, 9)
test_9mer  = is_nmer(X_test_3d, 9)

X_train_9 = X_train_3d[train_9mer]
X_test_9  = X_test_3d[test_9mer]
y_train_9 = y_train[train_9mer]
y_test_9  = y_test[test_9mer]

print(f'9-mer train: {X_train_9.shape}  positive rate {y_train_9.mean():.3f}')
print(f'9-mer test:  {X_test_9.shape}   positive rate {y_test_9.mean():.3f}')

9-mer train: (20690, 11, 21)  positive rate 0.317
9-mer test:  (5166, 11, 21)   positive rate 0.312


## Anchor positions in 9-mers (right-padded layout)

For a 9-mer in our 11-slot encoding:

- P1 → index 0  (N-terminal residue)
- **P2 → index 1**  ← primary N-terminal anchor (mask this)
- P3 → index 2  (sometimes secondary anchor)
- P4 → index 3  (TCR contact)
- P5 → index 4  (TCR contact)
- P6 → index 5  (TCR contact, sometimes secondary anchor)
- P7 → index 6  (TCR contact)
- P8 → index 7  (TCR contact)
- **P9 → index 8**  ← C-terminal anchor PΩ (mask this)
- index 9, 10 → padding

We will define `ANCHOR_INDICES = [1, 8]` and mask both.

In [4]:
ANCHOR_INDICES = [1, 8]  # P2 and PΩ for 9-mers in the right-padded encoding

## TODO #1: build the masked feature matrices

**Your task.** Write the function `mask_anchor_positions` below, then apply it to both `X_train_9` and `X_test_9` to produce `X_train_9_masked` and `X_test_9_masked`.

**Specification:**

- Input: an array shaped `(n_samples, MAX_LEN, N_AA)` and a list of position indices to mask.
- Output: a copy of the input where, for every sample and every position in the mask list, the entire 21-dim row is set to zero.
- The shape must be unchanged. Other positions must be untouched.
- Do not mutate the input; return a new array. (`.copy()` before assignment is the easy way.)

**Why fully zero, not pad?** A row of all zeros is the same trick the original encoding uses for unspecified content (well, the original uses the padding column, but for the model the effect is similar: no amino acid identity at that position). Using all zeros rather than "padding column on" keeps the comparison clean and makes the masked positions truly information-free.

**Sanity check after you implement it:** for each masked sample, the slice `masked[i, ANCHOR_INDICES, :].sum()` should equal 0.0.

In [5]:
def mask_anchor_positions(X_3d, positions_to_mask):
    """Zero out the given positions in a (n, MAX_LEN, N_AA) one-hot array."""
    X_masked = X_3d.copy()
    X_masked[:, positions_to_mask, :] = 0
    return X_masked

X_train_9_masked = mask_anchor_positions(X_train_9, ANCHOR_INDICES)
X_test_9_masked  = mask_anchor_positions(X_test_9,  ANCHOR_INDICES)

# Sanity check: confirm the masked columns are zero and others are untouched.
assert X_train_9_masked[:, ANCHOR_INDICES, :].sum() == 0
assert X_train_9_masked[:, [0, 2, 3, 4, 5, 6, 7], :].sum() == X_train_9[:, [0, 2, 3, 4, 5, 6, 7], :].sum()
print(f"Masked: train {X_train_9_masked.shape}, test {X_test_9_masked.shape}")
print("Sanity checks passed.")

Masked: train (20690, 11, 21), test (5166, 11, 21)
Sanity checks passed.


## Training helpers

Filled in for you. These functions wrap the same training procedure used in notebook 03 so the only thing varying between the two runs is the input. Each returns a small dict of metrics.

In [6]:
def train_lr(X_train_flat, y_train, X_test_flat, y_test):
    m = LogisticRegression(max_iter=1000, class_weight='balanced',
                           solver='lbfgs', random_state=RANDOM_STATE)
    m.fit(X_train_flat, y_train)
    p = m.predict_proba(X_test_flat)[:, 1]
    return {'acc': accuracy_score(y_test, (p >= 0.5).astype(int)),
            'auc': roc_auc_score(y_test, p)}

def train_rf(X_train_flat, y_train, X_test_flat, y_test):
    m = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train_flat, y_train)
    p = m.predict_proba(X_test_flat)[:, 1]
    return {'acc': accuracy_score(y_test, (p >= 0.5).astype(int)),
            'auc': roc_auc_score(y_test, p)}

def train_cnn(X_train_3d, y_train, X_test_3d, y_test):
    classes = np.array([0, 1])
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    cw = {0: weights[0], 1: weights[1]}

    inputs = keras.Input(shape=(MAX_LEN, N_AA))
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(inputs)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = keras.Model(inputs, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                       restore_best_weights=True)
    m.fit(X_train_3d, y_train, validation_split=0.1, epochs=50,
          batch_size=256, class_weight=cw, callbacks=[es], verbose=0)
    p = m.predict(X_test_3d, verbose=0).flatten()
    return {'acc': accuracy_score(y_test, (p >= 0.5).astype(int)),
            'auc': roc_auc_score(y_test, p)}

## Baseline: full 9-mer encoding

We re-train on **only the 9-mers** (not the full 47k mixed-length set) so the comparison against the masked version is apples-to-apples. The numbers here will be slightly different from notebook 04, which trained on all lengths.

In [7]:
X_train_9_flat = X_train_9.reshape(len(X_train_9), -1)
X_test_9_flat  = X_test_9.reshape(len(X_test_9), -1)

results_full = {
    'Logistic Regression': train_lr(X_train_9_flat, y_train_9, X_test_9_flat, y_test_9),
    'Random Forest':       train_rf(X_train_9_flat, y_train_9, X_test_9_flat, y_test_9),
    'CNN':                 train_cnn(X_train_9, y_train_9, X_test_9, y_test_9),
}
pd.DataFrame(results_full).T.round(4)

,acc,auc
Logistic Regression,0.5960,0.6366
Random Forest,0.7331,0.6725
CNN,0.6177,0.6366


## TODO #2: ablated run

**Your task.** Once your `mask_anchor_positions` function works:

1. Build `X_train_9_masked` and `X_test_9_masked` (uncomment the lines in TODO #1).
2. Flatten each and call the three training helpers exactly as in the cell above.
3. Store the results in a dict called `results_masked` with the same keys.
4. Build `comparison` below and print it.

Skeleton is provided; replace the `NotImplementedError`-raising placeholder.

In [8]:
results_masked = {
    'Logistic Regression': train_lr(X_train_9_masked.reshape(len(X_train_9_masked), -1),
                                    y_train_9,
                                    X_test_9_masked.reshape(len(X_test_9_masked), -1),
                                    y_test_9),
    'Random Forest':       train_rf(X_train_9_masked.reshape(len(X_train_9_masked), -1),
                                    y_train_9,
                                    X_test_9_masked.reshape(len(X_test_9_masked), -1),
                                    y_test_9),
    'CNN':                 train_cnn(X_train_9_masked, y_train_9, X_test_9_masked, y_test_9),
}
pd.DataFrame(results_masked).T.round(4)

,acc,auc
Logistic Regression,0.5823,0.6123
Random Forest,0.7251,0.6443
CNN,0.6144,0.6128


In [9]:
comparison = pd.DataFrame({
    'full_auc':   {k: v['auc'] for k, v in results_full.items()},
    'masked_auc': {k: v['auc'] for k, v in results_masked.items()},
})
comparison['delta'] = comparison['masked_auc'] - comparison['full_auc']
comparison.round(4)

,full_auc,masked_auc,delta
Logistic Regression,0.6366,0.6123,-0.0243
Random Forest,0.6725,0.6443,-0.0283
CNN,0.6366,0.6128,-0.0239


## How to read your result

Once you have the `comparison` table, write a short paragraph below it interpreting the numbers in light of the three outcomes from the introduction. Specifically:

1. **What is the magnitude of the AUC drop (or gain)?** A change of 0.01-0.02 is within noise on a single split; a change of 0.05+ is meaningful; a change of 0.10+ is dramatic.
2. **Is the drop similar across all three models, or one-sided?** If only one model drops, that model was relying on anchors more than the others. If all drop similarly, the anchor information is genuinely useful regardless of model class.
3. **Cross-reference notebook 05's heatmap.** If the LR coefficients showed strong anchor signal at P2 and P9 and the masked AUC drops a lot, the two pieces of evidence are consistent. If the heatmap showed strong anchor signal but masking *doesn't* hurt, that means the model could route around the missing information using correlated features.

## Optional stretch goals

If this experiment was easy and you want more:

- **The reverse experiment.** Mask P4-P7 (TCR contacts) instead of anchors. Does that hurt more or less than masking the anchors?
- **Mask one anchor at a time.** P2 only, then P9 only. Does the model rely on them equally?
- **Center-padding redo.** Re-encode the full 8-11mer set so that PΩ always lands in the same column, then run the same ablation across all lengths. This is the principled version.
- **Physicochemical encoding.** Replace each amino acid with a vector of (hydrophobicity, charge, molecular weight, side-chain volume), typically 4-5 dims instead of 21. Compare AUC to the one-hot baseline. This is a separate experiment but uses the same pipeline.